# M6 — OCR loop on real kid pages + measured held-out handwriting accuracy

Implements PROJECT_BRIEF C4 (mtmd vision path) and acceptance criterion **#3**
(OCR character accuracy >= 85% on the held-out real-kid-handwriting set).

| Cell | What | Gate | Resume flag |
|------|------|------|-------------|
| C1 | setup: root, config, resume helpers, char-accuracy metric | - | - |
| A | acquire OCR brain (llama-server > merged MLX dir > labelled pending) | - | `A` |
| B | OCR the 2 real kid homework pages | non-empty + sane | `B` |
| C | held-out handwriting eval (60 samples) vs ground truth | char acc >= 85% | `C` |
| D | CROHME held-out (200 math expressions) — reported, no hard gate | - | `D` |
| E | baseline comparison vs M4 base-model smoke (outputs/m4/ocr_smoke_base.json) | - | `E` |
| F | summary -> outputs/m6/ocr_report.json | - | `F` |

The kid-confirm step (M7) is the product safety net for OCR errors; this
notebook measures how often it will be needed. If no GEETA weights exist yet
(M4 pending), cells B-F run in `pending-M4` mode: they record the plan but
skip measurement instead of faking numbers.

In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501

In [ ]:

# ---- C1 setup: root, config, resume helpers ---------------------------------
import json, os, time, hashlib
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "PROJECT_BRIEF.md").exists():
    for cand in (ROOT, ROOT.parent, ROOT.parent.parent):
        if (cand / "PROJECT_BRIEF.md").exists():
            ROOT = cand
            break
os.chdir(ROOT)
assert (ROOT / "PROJECT_BRIEF.md").exists(), "cannot locate PROJECT_BRIEF.md from cwd=" + str(Path.cwd())

import yaml
CFG = yaml.safe_load(open("config/config.yaml"))
OUT = ROOT / "outputs" / "m6"
OUT.mkdir(parents=True, exist_ok=True)

def done(flag):
    return (OUT / ("." + flag + ".done")).exists()

def mark(flag, payload=None):
    p = OUT / ("." + flag + ".done")
    if not p.exists():
        p.write_text(json.dumps({"flag": flag, "ts": time.time(), "payload": payload}) + chr(10))
    print("[m6] flagged", flag, "done")

def log(name, payload):
    path = OUT / name
    path.write_text(json.dumps(payload, indent=2, default=str))
    print("[m6] wrote", path)

import re, unicodedata

_PUNCT = re.compile(r"[\s\.,;:!?'\u2018\u2019\"()\[\]{}=]+")
def norm(s):
    s = unicodedata.normalize("NFKC", str(s)).casefold()
    return _PUNCT.sub("", s)

def levenshtein(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def char_accuracy(pred, gt):
    """1 - normalised character error rate (Levenshtein / len(gt))."""
    p, g = norm(pred), norm(gt)
    if not g:
        return 1.0 if not p else 0.0
    return max(0.0, 1.0 - levenshtein(p, g) / len(g))

# sanity-check the metric on a known pair
assert char_accuracy("Round 99 to the nearest 100", "round 99 to the nearest 100") > 0.99
assert char_accuracy("Round 99 to the nearest 10", "Round 99 to the nearest 100") < 0.96   # 1-char error in 22-char string = 0.9545
print("[m6] char-accuracy metric self-test OK")

# OCR brain + source are set by cell A; defaults keep later cells safe on skip.
ocr_model = None
OCR_SOURCE = "pending-M4"

def ocr_image(path):
    """Transcription text for one image via the active OCR brain."""
    if ocr_model is None:
        return ""
    prompt = ("Transcribe this maths homework exactly as written. Output only the "
              "transcription, one question per line, no commentary.")
    if ocr_model[0] == "http":
        return ocr_model[1].ocr_page(path)
    import mlx_vlm
    from PIL import Image
    img = Image.open(path).convert("RGB")
    out = mlx_vlm.generate(ocr_model[1], ocr_model[2], [prompt], images=[img],
                           max_tokens=CFG["serving"]["max_tokens"])
    if isinstance(out, list):
        out = out[0]
    if isinstance(out, dict):
        out = out.get("response") or out.get("text") or str(out)
    return str(out)

def done_real(flag):
    """True only if ``flag`` was marked in a run with a REAL OCR brain.
    pending-M4 runs never block a later real re-run (source-aware guards)."""
    p = OUT / ("." + flag + ".done")
    if not p.exists():
        return False
    src = (json.loads(p.read_text()).get("payload") or {}).get("ocr_source", "")
    return src.startswith(("real:", "mlx:"))

In [ ]:

# ---- Cell A: acquire OCR brain -------------------------------------------------
if done_real("A"):
    print("A already done with real OCR brain - skipping")
else:
    import requests

    base = CFG["model"]["server_base_url"]
    try:
        alive = requests.get(base + "/health", timeout=3).ok
    except requests.RequestException:
        alive = False

    ocr_model = None
    if alive:
        from geeta.model_client import ModelClient
        ocr_model = ("http", ModelClient(base, max_tokens=CFG["serving"]["max_tokens"]))
        OCR_SOURCE = "real:" + base
    elif (ROOT / "models" / "geeta-4b-mlx").exists():
        from mlx_vlm import utils as vutils
        _m, _p = vutils.load(str(ROOT / "models" / "geeta-4b-mlx"))
        ocr_model = ("mlx", _m, _p)
        OCR_SOURCE = "mlx:models/geeta-4b-mlx"
    else:
        OCR_SOURCE = "pending-M4"

    print("[m6] OCR source:", OCR_SOURCE)
    mark("A", {"ocr_source": OCR_SOURCE})

In [ ]:

# ---- Cell B: OCR the 2 real kid homework pages ---------------------------------
if done_real("B"):
    print("B already done with real OCR brain - skipping")
else:
    pages = sorted((ROOT / "data" / "processed" / "pages").glob("*.png"))
    results = []
    for pg in pages:
        t0 = time.time()
        text = ocr_image(str(pg))
        dt = time.time() - t0
        ok = len(text.strip()) > 20
        results.append({"path": str(pg.relative_to(ROOT)), "ok": ok,
                         "chars": len(text), "seconds": round(dt, 1),
                         "output_head": text[:200]})
        print("[m6] %s: %d chars in %.1fs" % (pg.name, len(text), dt))
    log("real_pages.json", {"ocr_source": OCR_SOURCE, "results": results})
    if ocr_model is not None:
        assert all(r["ok"] for r in results), "OCR gate failed on real pages"
    mark("B", {"ocr_source": OCR_SOURCE, "n_pages": len(results),
               "all_nonempty": all(r["ok"] for r in results)})

In [ ]:

# ---- Cell C: held-out handwriting eval (60 samples) ----------------------------
if done_real("C"):
    print("C already done with real OCR brain - skipping")
else:
    manifest = json.load(open("data/eval/handwriting_manifest.json"))
    eval_rows = [r for r in manifest.get("eval", [])]
    n = int(os.environ.get("GEETA_OCR_N", "0")) or len(eval_rows)
    rows = eval_rows[:n]
    results, accs = [], []
    for r in rows:
        img = ROOT / r["image"]
        if not img.exists():
            continue
        t0 = time.time()
        pred = ocr_image(str(img))
        acc = char_accuracy(pred, r["transcription"])
        accs.append(acc)
        results.append({"image": r["image"], "gt": r["transcription"],
                         "pred_head": pred[:120], "char_acc": round(acc, 4),
                         "seconds": round(time.time() - t0, 1)})
    mean_acc = sum(accs) / len(accs) if accs else None
    summary_c = {"ocr_source": OCR_SOURCE, "n_samples": len(accs),
                 "char_accuracy_mean": round(mean_acc, 4) if mean_acc is not None else None,
                 "gate_85": (mean_acc >= 0.85) if mean_acc is not None else None,
                 "results": results}
    log("handwriting_eval.json", summary_c)
    if ocr_model is not None:
        assert mean_acc is not None and mean_acc >= 0.85, \
            "OCR gate FAILED: char acc %.1f%% < 85%%" % (100 * mean_acc)
    mark("C", {"char_accuracy_mean": summary_c["char_accuracy_mean"]})
    print("[m6] handwriting char acc: %s" %
          ("%.1f%%" % (100 * mean_acc) if mean_acc is not None else "pending M4"))

In [ ]:

# ---- Cell D: CROHME held-out (200 math expressions) — reported, no gate --------
if done_real("D"):
    print("D already done with real OCR brain - skipping")
else:
    crohme = [json.loads(l) for l in open("data/eval/crohme_heldout.jsonl")]
    n = int(os.environ.get("GEETA_OCR_N", "0")) or len(crohme)
    rows = crohme[:n]
    results, accs = [], []
    for i, r in enumerate(rows):
        img = ROOT / r["image"]
        if not img.exists():
            continue
        pred = ocr_image(str(img))
        acc = char_accuracy(pred, r["label"])
        accs.append(acc)
        results.append({"image": r["image"], "gt_head": r["label"][:80],
                         "pred_head": pred[:80], "char_acc": round(acc, 4)})
        if (i + 1) % 50 == 0:
            print("[m6] crohme %d/%d mean acc %.1f%%" % (i + 1, len(rows),
                  100.0 * sum(accs) / len(accs)))
    mean_acc = sum(accs) / len(accs) if accs else None
    log("crohme_eval.json", {"ocr_source": OCR_SOURCE, "n_samples": len(accs),
                              "char_accuracy_mean": round(mean_acc, 4) if mean_acc is not None else None,
                              "results": results})
    mark("D", {"char_accuracy_mean": mean_acc})
    print("[m6] CROHME char acc: %s" %
          ("%.1f%%" % (100 * mean_acc) if mean_acc is not None else "pending M4"))

In [ ]:

# ---- Cell E: baseline comparison vs M4 base-model smoke -------------------------
if done_real("E"):
    print("E already done with real OCR brain - skipping")
else:
    base_p = ROOT / "outputs" / "m4" / "ocr_smoke_base.json"
    base = json.load(open(base_p)) if base_p.exists() else None
    mine = json.load(open(OUT / "real_pages.json")) if (OUT / "real_pages.json").exists() else None
    comparison = {
        "baseline_source": "outputs/m4/ocr_smoke_base.json (BASE 8-bit model, pre-finetune)",
        "baseline": base.get("results") if base else None,
        "geeta": mine.get("results") if mine else None,
        "note": "same 2 real pages; compare chars/seconds/output_head",
    }
    log("baseline_comparison.json", comparison)
    mark("E", {"baseline_available": base is not None})

In [ ]:

# ---- Cell F: M6 summary ----------------------------------------------------------
if done_real("F"):
    print("F already done with real OCR brain - skipping")
else:
    def _load(name):
        p = OUT / name
        return json.load(open(p)) if p.exists() else {}

    hw = _load("handwriting_eval.json")
    cr = _load("crohme_eval.json")
    rp = _load("real_pages.json")
    summary = {
        "milestone": "M6", "ocr_source": OCR_SOURCE,
        "handwriting_char_accuracy": hw.get("char_accuracy_mean"),
        "handwriting_gate_85": hw.get("gate_85"),
        "crohme_char_accuracy": cr.get("char_accuracy_mean"),
        "real_pages": rp.get("results"),
        "acceptance": {"#3 OCR char accuracy >= 85% (held-out handwriting)": hw.get("gate_85")},
        "note": ("pending M4 - no measured numbers" if OCR_SOURCE == "pending-M4"
                 else "measured on " + OCR_SOURCE),
    }
    log("ocr_report.json", summary)
    mark("F", {"ocr_source": OCR_SOURCE})
    print(json.dumps(summary, indent=2))